The scores confirm that **Trial 3258** remains the most robust high-scoring model locally, with the smallest drop in performance on the unseen Kaggle data.

## 👑 Optuna Hyperparameter and Performance Comparison

| Trial Number | Validation Score (Optuna PS) | **Kaggle Score (Kaggle PS)** | **Generalization Delta** ($\text{Kaggle} - \text{Validation}$) | $\mathbf{ml\_conf\_factor}$ | Notes |
| :---: | :---: | :---: | :---: | :---: | :--- |
| **397** (Historical) | $1.01052$ | $\mathbf{1.112}$ | $+\mathbf{0.10148}$ | $4.5365$ | **Historical Best Score** (Lucky/Unstable Generalization). |
| **957** | $1.02287$ | N/A | N/A | $6.9385$ | Initial Local Best (Higher validation score than 397). |
| **2239** | $1.00985$ | N/A | N/A | $6.3107$ | First $V>1.0$ post-resume. |
| **3104** | $1.01422$ | N/A | N/A | $4.9432$ | Stable but lower confidence. |
| **3258** | **$\mathbf{1.06937}$** | $\mathbf{1.053}$ | **$-0.01637$** | $6.6209$ | **Most Stable High-Performer** (Smallest drop in performance). |
| **3319** | $1.03807$ | $\mathbf{0.994}$ | **$-0.04407$** | $6.5662$ | Overfit more significantly than Trial 3258. |
| **3455** | $1.03546$ | N/A | N/A | $6.7466$ | Result Pending (Expected $K \approx 0.99$). |

This represents the peak of performance you can reliably expect from your current feature engineering approach.

***

## 🚀 Final Competition Configuration Summary

| Setting | Value | Rationale |
| :--- | :--- | :--- |
| **Model** | 3-Model Ensemble (XGB + LGBM + CatBoost) | Proven ensemble structure from Optuna. |
| **Hyperparameters** | **Trial 3258** | Achieved the best validation score ($\mathbf{1.069}$) with the **minimal generalization drop** ($\mathbf{-0.01637}$) on the Kaggle test set. |
| **Feature Set** | **Restored 1187 Features** | Confirmed to be the optimal set; the expanded features introduced too much noise, dropping the score significantly to $0.92$. |
| **Kaggle Score** | $\mathbf{1.053}$ | The proven, repeatable out-of-sample performance. |

This code is now finalized and running in its highest-confidence state.

The XGBoost model was the most heavily weighted component of the final ensemble (weight of $4.183$), and its specific hyperparameters, as determined by Optuna Trial 3258, are summarized below:

### ⚙️ XGBoost Hyperparameters (Trial 3258)

The model was configured to run a gradient boosting classification routine using these settings:

| Parameter | Code Variable | Value | Description |
| :--- | :--- | :--- | :--- |
| **Maximum Depth** | `XGB_MAX_DEPTH` | $14$ | The maximum depth of each boosting tree. A high value suggests the model is allowed to capture complex, non-linear interactions. |
| **Learning Rate** | `XGB_LR` | $0.03260$ | The step size shrinkage used in updates to prevent overfitting. This is a small, conservative rate. |
| **L1 Regularization** | `XGB_REG_ALPHA` | $0.17226$ | The L1 term on weights (Lasso regularization). Helps prune less important features. |
| **L2 Regularization** | `XGB_REG_LAMBDA` | $10.52825$ | The L2 term on weights (Ridge regularization). A high value strongly smooths the final weights. |
| **Row Subsample** | `XGB_SUBSAMPLE` | $0.80185$ | The fraction of the training data randomly sampled for each tree. Helps prevent overfitting. |
| **Column Subsample** | `XGB_COLSAMPLE` | $0.72828$ | The fraction of columns (features) randomly sampled for each tree. Helps prevent overfitting and improves diversity. |

**Training Summary:**
* **Objective:** `binary:logistic` (predicts the probability of a positive return).
* **Evaluation Metric:** `logloss` (used during training and early stopping).
* **Tree Method:** `hist` (Histogram-based algorithm for faster training).
* **Total Estimators:** Up to $5000$ iterations, but stopped early at $18$ trees during validation (due to `EARLY_STOPPING_ROUNDS = 50`).

***

 Here are the specific hyperparameters used for the **LightGBM** and **CatBoost** models in the final, optimized ensemble (Trial 3258).

### ⚙️ LightGBM Hyperparameters

LightGBM was the lowest-weighted model in the ensemble (weight of $0.296$), suggesting it was used to provide a slightly diverse, less dominant perspective.

| Parameter | Code Variable | Value | Description |
| :--- | :--- | :--- | :--- |
| **Maximum Depth** | `LGB_MAX_DEPTH` | $8$ | A relatively shallow depth, which helps prevent overfitting and keeps the model computationally efficient. |
| **Learning Rate** | `LGB_LR` | $0.01396$ | A very small learning rate, favoring a slow, careful convergence over many small steps. |
| **Number of Leaves** | `LGB_NUM_LEAVES` | $141$ | The number of terminal nodes in a tree. The default is 31; $141$ is significantly higher and suggests a more complex tree structure than the default. |
| **L1 Regularization** | `LGB_L1` | $0.92451$ | The L1 penalty on the weights (or leaves). Used to regularize the model and encourage feature sparsity. |

**Training Summary:**
* **Objective:** `binary` (predicts the probability of a positive return).
* **Device Type:** `gpu` (leveraged GPU for faster training).
* **Total Estimators:** Up to $5000$ iterations, but stopped early at only $1$ tree during validation, suggesting the model may have either converged very quickly or the early stopping criteria were triggered immediately by a lack of improvement.

### ⚙️ CatBoost Hyperparameters

CatBoost had the second-highest weight (weight of $1.928$), making it a significant contributor to the final prediction.

| Parameter | Code Variable | Value | Description |
| :--- | :--- | :--- | :--- |
| **Tree Depth** | `CBT_DEPTH` | $11$ | The maximum depth of the trees. This is a moderate depth, balancing complexity and generalization. |
| **Learning Rate** | `CBT_LR` | $0.03679$ | A small learning rate, similar to XGBoost, for stable convergence. |
| **L2 Regularization** | `CBT_L2_REG` | $4.60131$ | The L2 term on the leaf values. This strong regularization helps to prevent overfitting. |
| **Bootstrap Type** | `bootstrap_type` | `Bayesian` | CatBoost's specific method for sampling weights and preventing target leakage. |

**Training Summary:**
* **Loss/Eval Function:** `Logloss`.
* **Task Type:** `GPU` (leveraged GPU for faster training).
* **Total Estimators:** Up to $5000$ iterations, but stopped very early at only $3$ iterations, similar to LightGBM.

This ensemble, dominated by the heavily-weighted, deeper XGBoost model (Depth 14) and the robust CatBoost model (Depth 11), was selected for its best combination of a high validation score ($\mathbf{1.069}$) and minimal generalization drop.

In [1]:
# FINAL VERSION - STABILIZED ENSEMBLE WITH RESTORED SCORING (v4)
import os
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import pandas.api.types
from itertools import product
import warnings
import logging
import lightgbm as lgb
import platform
import sys
from contextlib import contextmanager 

# --- Setup and Logging ---
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', module='lightgbm')

os.environ['CATBOOST_DRIVER_COMPATIBLE'] = '1'
os.environ['CATBOOST_QUIET_MODE'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# ===========================================================================
# WARNING SUPPRESSION CONTEXT MANAGER
# ===========================================================================
@contextmanager
def suppress_stderr():
    """Temporarily redirect stderr to devnull to suppress native C warnings."""
    original_stderr = sys.stderr
    try:
        # Redirect stderr to /dev/null
        with open(os.devnull, 'w') as f:
            sys.stderr = f
            yield
    finally:
        # Restore original stderr
        sys.stderr = original_stderr


# ===========================================================================
# OFFICIAL KAGGLE METRIC (RESTORED)
# ===========================================================================
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = 'row_id') -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ValueError('Predictions must be numeric')

    sol = solution.copy()
    sol['position'] = submission['prediction'].values

    if sol['position'].max() > MAX_INVESTMENT:
        raise ValueError(f'Position exceeds {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ValueError(f'Position below {MIN_INVESTMENT}')

    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    strategy_excess = sol['strategy_returns'] - sol['risk_free_rate']
    strategy_cum = (1 + strategy_excess).prod()
    strategy_mean = strategy_cum ** (1 / len(sol)) - 1
    strategy_std = sol['strategy_returns'].std()
    trading_days = 252

    if strategy_std == 0:
        return 0.0
    sharpe = strategy_mean / strategy_std * np.sqrt(trading_days)

    strategy_vol = float(strategy_std * np.sqrt(trading_days) * 100)

    market_excess = sol['forward_returns'] - sol['risk_free_rate']
    market_cum = (1 + market_excess).prod()
    market_mean = market_cum ** (1 / len(sol)) - 1
    market_std = sol['forward_returns'].std()
    market_vol = float(market_std * np.sqrt(trading_days) * 100)

    if market_vol == 0:
        return 0.0

    excess_vol = max(0, strategy_vol / market_vol - 1.2)
    vol_penalty = 1 + excess_vol

    return_gap = max(0, (market_mean - strategy_mean) * 100 * trading_days)
    return_penalty = 1 + (return_gap ** 2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


# ===========================================================================
# FEATURE ENGINEERING FUNCTION (POLARS - RESTORED ORIGINAL SET)
# ===========================================================================
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df_copy = df.clone()
    potential_base_feature_prefixes = ('M','E','I','P','V','S')
    all_potential_features = [
        c for c in df_copy.columns
        if c.startswith(potential_base_feature_prefixes) and c != 'market_forward_excess_returns'
    ]
    casting_expressions = []
    for c in all_potential_features:
        casting_expressions.append(
            pl.col(c).cast(pl.Float64, strict=False).alias(c)
        )
    if casting_expressions:
        df_copy = df_copy.with_columns(casting_expressions)
    base_features = [
        c for c in all_potential_features
        if df_copy.schema.get(c) in pl.NUMERIC_DTYPES
    ]
    expressions = []

    # --- 1. Lags (Original: 4 Lags) ---
    for c in base_features:
        for lag in [1, 2, 5, 10]:
            expressions.append(
                pl.col(c).shift(lag).over('date_id').fill_null(0).alias(f'{c}_L{lag}')
            )
    
    # --- 2. Rolling Window Features (Original: 2 Windows, 3 Stats) ---
    for c in base_features:
        for w in [5, 10]:
            expressions.append(
                pl.col(c).rolling_mean(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMean{w}')
            )
            expressions.append(
                pl.col(c).rolling_std(window_size=w, min_periods=1).over('date_id').fill_nan(0).fill_null(0).alias(f'{c}_RStd{w}')
            )
            expressions.append(
                pl.col(c).rolling_max(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMax{w}')
            )
            
    df_copy = df_copy.with_columns(expressions)
    expressions = []
    
    # --- 3. Rank and Z-Score ---
    for c in base_features:
        expressions.append(
            pl.col(c).rank(method='min').over('date_id').fill_null(0).alias(f'{c}_RANK')
        )
        mean_c = pl.col(c).mean().over('date_id')
        std_c  = pl.col(c).std().over('date_id')
        std_c_safe_expr = pl.when(pl.col(c).is_null() | std_c.is_null() | (std_c == 0)).then(1e-6).otherwise(std_c)
        expressions.append(
            ((pl.col(c).fill_null(mean_c) - mean_c) / std_c_safe_expr).fill_nan(0).fill_null(0).alias(f'{c}_ZSCORE')
        )
    df_copy = df_copy.with_columns(expressions)
    rank_cols = [f'{c}_RANK' for c in base_features]
    zscore_cols = [f'{c}_ZSCORE' for c in base_features]
    expressions = []

    # --- 4. Interactions (Original Set only) ---
    
    # Original Targeted Interactions (M4, M1, E1, V2)
    for c in ['M4','M1','E1','V2']:
        r = f'{c}_RANK'
        if f'{c}' in df_copy.columns and r in df_copy.columns:
             expressions.append((pl.col(c) * pl.col(r)).alias(f'{c}_x_{r}'))
             expressions.append((pl.col(c) / (pl.col(r) + 1e-6)).alias(f'{c}_div_{r}'))

    target_rank_cols = rank_cols[:12]
    # 1. Rank * Rank (Unique Pairs) - Adds 66 features
    for r_col1, r_col2 in product(target_rank_cols, target_rank_cols):
        c1 = r_col1.split('_')[0]
        c2 = r_col2.split('_')[0]
        if c1 < c2:
            expressions.append((pl.col(r_col1) * pl.col(r_col2)).fill_nan(0).fill_null(0).alias(f'{c1}R_x_{c2}R'))
    # 2. Rank * ZScore - Adds 9 features
    target_zscore_cols = zscore_cols[:9]
    for r_col, z_col in zip(rank_cols[:9], target_zscore_cols):
        expressions.append((pl.col(r_col) * pl.col(z_col)).fill_nan(0).fill_null(0).alias(f'{r_col}_x_{z_col}'))

    if expressions:
        df_copy = df_copy.with_columns(expressions)
    return df_copy


# ===========================================================================
# DATA LOADING AND SPLITTING
# ===========================================================================
logger.info("Loading data and splitting for validation...")
try:
    train_full_pl = pl.read_csv(DATA_PATH / "train.csv")
except FileNotFoundError:
    logger.error('Could not find \'train.csv\'. Please ensure the DATA_PATH is correct.')
    raise

train_full_pd = train_full_pl.to_pandas()

split_idx = int(len(train_full_pd) * 0.8)
train_pd = train_full_pd.head(split_idx).copy()
val_pd   = train_full_pd.tail(len(train_full_pd) - split_idx).copy()
full_train_pd = train_full_pd.copy() # Keep full copy for models that need 100% data

# --- Revert to Classification Target (Required for Optimal Score) ---
train_pd['target_binary'] = (train_pd['market_forward_excess_returns'] > 0).astype(np.int8)
val_pd['target_binary'] = (val_pd['market_forward_excess_returns'] > 0).astype(np.int8)
full_train_pd['target_binary'] = (full_train_pd['market_forward_excess_returns'] > 0).astype(np.int8)


train_pl = pl.from_pandas(train_pd)
val_pl   = pl.from_pandas(val_pd)
full_train_pl = pl.from_pandas(full_train_pd)

logger.info(f"✅ Oracle Dictionary ready. Binary Target created.")


# ===========================================================================
# ENSEMBLE PARAMETERS (UPDATED TO OPTUNA TRIAL 3258 - Score 1.06937)
# ===========================================================================

# --- OPTUNA TRIAL 3258 BEST (Score 1.06937) ---
BEST_W_XGB = 4.183022496349415     # XGBoost Weight
BEST_W_LGB = 0.29640061148676144   # LightGBM Weight
BEST_W_CAT = 1.9283230458964578   # CatBoost Weight
BEST_W_LOGREG = 0.0 
ML_CONF_FACTOR = 6.6208527134892385 # Optimal Risk Factor

# Model-specific Hyperparameters (Trial 3258)
XGB_MAX_DEPTH = 14
XGB_LR = 0.03260054929298955
XGB_REG_ALPHA = 0.1722628424051081
XGB_REG_LAMBDA = 10.528253815338909
XGB_SUBSAMPLE = 0.8018511121220447
XGB_COLSAMPLE = 0.7282877621162328

LGB_MAX_DEPTH = 8
LGB_LR = 0.013960942809215957
LGB_NUM_LEAVES = 141
LGB_L1 = 0.9245109459111609

CBT_DEPTH = 11
CBT_LR = 0.03679836629440696
CBT_L2_REG = 4.601313084093791

# Fixed Estimators and Stopping
N_ESTIMATORS_MAX = 5000
XGB_ES = 200 # Use a moderate patience for the dominant model
LGB_CAT_ES = N_ESTIMATORS_MAX # Ensure full train for unstable models


logger.info(f"Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:{BEST_W_XGB:.2f}, LGB:{BEST_W_LGB:.2f}, CAT:{BEST_W_CAT:.2f}")
logger.info(f"Using OPTUNA BEST ENSEMBLE (Trial 3258), Risk Factor: {ML_CONF_FACTOR:.4f}")
logger.info(f"Using Strategic Stopping: XGB:{XGB_ES} (Val Check), LGB/CAT:{LGB_CAT_ES} (Full Train)")


# ===========================================================================
# ENSEMBLE PIPELINE (MAIN PIPELINE)
# ===========================================================================
logger.info("Training ensemble (Expanded Feature Set)...")

# 1. Feature Engineering 
train_engineered_pl = create_features(train_pl)
val_engineered_pl = create_features(val_pl)
full_train_engineered_pl = create_features(full_train_pl)

# 2. Feature Column Selection 
feature_cols = []
original_suffixes = ('_L1', '_L2', '_L5', '_L10', '_RMean5', '_RStd5', '_RMax5', '_RMean10', '_RStd10', '_RMax10', '_RANK', '_ZSCORE', '_x_RANK', '_div_RANK', '_x_R', '_x_ZSCORE')

for col in train_engineered_pl.columns:
    is_base_feature = col.startswith(('M', 'E', 'I', 'P', 'V', 'S'))
    is_engineered_feature = any(col.endswith(s) for s in original_suffixes)
    
    if (is_base_feature or is_engineered_feature) and (train_engineered_pl[col].null_count() / len(train_engineered_pl) < 0.95):
        feature_cols.append(col)

feature_cols = [c for c in feature_cols if c not in ['market_forward_excess_returns', 'target_binary', 'forward_returns', 'date_id']]
FINAL_FEATURE_COLS = feature_cols 
logger.info(f"Features: {len(FINAL_FEATURE_COLS)} (Restored Optimized Set)") 

# 3. Data Preparation for ML
# XGBoost Data (80% train, 20% val for ES)
X_train = train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_train = train_pl['target_binary'].fill_null(0).to_numpy().astype(int)
X_val = val_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_val = val_pl['target_binary'].fill_null(0).to_numpy().astype(int)

# LGB/CAT Data (100% full train set)
X_full_train = full_train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_full_train = full_train_pl['target_binary'].fill_null(0).to_numpy().astype(int)


# Enforce Feature Order
X_train = X_train.loc[:, FINAL_FEATURE_COLS]
X_val = X_val.loc[:, FINAL_FEATURE_COLS]
X_full_train = X_full_train.loc[:, FINAL_FEATURE_COLS]


# Scaling (Fit only on 80% train data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_full_train_scaled = scaler.transform(X_full_train) # Transform full set for LGB prediction

# DataFrames for LGB and DMatrix for XGB
X_lgb = pd.DataFrame(X_train_scaled, columns=FINAL_FEATURE_COLS)
X_val_lgb = pd.DataFrame(X_val_scaled, columns=FINAL_FEATURE_COLS)
X_full_lgb = pd.DataFrame(X_full_train_scaled, columns=FINAL_FEATURE_COLS)

dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dval = xgb.DMatrix(X_val_scaled, label=y_val)


# 4. Model Training (3 Models + 1 inert model)
# --- WRAP MODEL TRAINING IN suppress_stderr() TO HIDE C-LEVEL WARNINGS ---
with suppress_stderr():
    # --- XGBoost (Trained on 80% data with ES) ---
    xgb_params = {
        'max_depth': XGB_MAX_DEPTH, 'eta': XGB_LR, 'alpha': XGB_REG_ALPHA, 'lambda': XGB_REG_LAMBDA, 
        'subsample': XGB_SUBSAMPLE, 'colsample_bytree': XGB_COLSAMPLE, 'seed': 42,
        'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss', 'verbosity': 0
    }

    xgb_model = xgb.train(
        xgb_params, dtrain, N_ESTIMATORS_MAX, evals=[(dval, 'val')],
        early_stopping_rounds=XGB_ES, verbose_eval=False
    )
    logger.info(f"XGBoost trained (Stopped at {xgb_model.best_iteration} trees).")

    # --- LightGBM (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    lgb_model = LGBMClassifier(
        n_estimators=N_ESTIMATORS_MAX, max_depth=LGB_MAX_DEPTH, learning_rate=LGB_LR, 
        num_leaves=int(LGB_NUM_LEAVES), lambda_l1=LGB_L1, random_state=123,
        objective='binary', metric='binary_logloss', n_jobs=-1, device_type='gpu',
        verbose=-1, _log_period=-1
    ).fit(
        X_full_lgb, y_full_train, # Train on 100% data
        # DO NOT pass eval_set or callbacks to force run to N_ESTIMATORS_MAX
    )
    # The number of estimators will be N_ESTIMATORS_MAX if it runs to completion
    logger.info(f"LGBMClassifier trained (Full 100% data, {lgb_model.n_estimators} trees).")


    # --- CatBoost (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    cbt_model = CatBoostClassifier(
        iterations=N_ESTIMATORS_MAX, depth=int(CBT_DEPTH), learning_rate=CBT_LR, 
        l2_leaf_reg=CBT_L2_REG, random_seed=789, loss_function='Logloss', eval_metric='Logloss',
        bootstrap_type='Bayesian', task_type="GPU", logging_level='Silent', allow_writing_files=False, 
        # DO NOT pass early_stopping_rounds or eval_set to force run to max iterations
    ).fit(
        X_full_train, y_full_train, # Train on 100% data
    )
    logger.info(f"CatBoostClassifier trained (Full 100% data, {cbt_model.get_best_iteration()} iterations).")

    # --- Logistic Regression (Trained on 80% data, but weighted 0.0) ---
    logreg_model = LogisticRegression(
        penalty='l2', C=0.01, solver='liblinear', max_iter=1000, random_state=999
    ).fit(X_lgb, y_train)
    logger.info(f"LogisticRegression trained.")

logger.info("✅ Ensemble trained.")
# --------------------------------------------------------------------------


# ===========================================================================
# VALIDATION – SCORE INTEGRATION (3-MODEL ENSEMBLE)
# ===========================================================================
logger.info('Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...')

# Constants are now locally accessible within this block for the scoring logic
# Prediction for XGBoost uses its early-stopped version (best_iteration) on the 20% validation set
prob_xgb = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration)) 
# Prediction for LGB/Cat uses the full 5000-iteration model on the 20% validation set
prob_lgb = lgb_model.predict_proba(X_val_lgb)[:, 1]
prob_cbt = cbt_model.predict_proba(X_val)[:, 1]
prob_logreg = logreg_model.predict_proba(X_val_lgb)[:, 1]

# Calculate final ensemble position using the 3 positive weights
total_w_final = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt + BEST_W_LOGREG * prob_logreg) / total_w_final

confidence = 2 * np.abs(avg_prob - 0.5)
positions_final = np.clip(confidence * ML_CONF_FACTOR, 0.0, 2.0)

submission_df_final = pd.DataFrame({'prediction': positions_final})

try:
    real_ps = score(val_pd, submission_df_final)
    logger.info(f'FINAL SUBMISSION PS SCORE ON VALIDATION (STABILIZED) = {real_ps:.6f}') 
except Exception as e:
    logger.error(f'Final Scoring error: {e}')
    real_ps = 0.0


# ===========================================================================
# PREDICT FUNCTION (FINAL ROBUST VERSION - 3-MODEL ENSEMBLE)
# ===========================================================================
def predict(test: pl.DataFrame) -> float:
    
    # Define constants locally for the predict function scope
    BEST_W_XGB = 4.183022496349415
    BEST_W_LGB = 0.29640061148676144
    BEST_W_CAT = 1.9283230458964578
    BEST_W_LOGREG = 0.0 
    ML_CONF_FACTOR = 6.6208527134892385 
    
    if xgb_model is None or lgb_model is None or cbt_model is None or logreg_model is None or scaler is None or not FINAL_FEATURE_COLS:
        return 0.0

    date_id = None
    try:
        date_id = int(test.select("date_id").to_series().item())
    except:
        pass
    
    try:
        test_engineered = create_features(test)
        X_test = test_engineered.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
        
        X_clean = X_test.loc[:, FINAL_FEATURE_COLS]
        X_clean = X_clean.fillna(0).replace([np.inf, -np.inf], 0) 

        X_scaled = scaler.transform(X_clean.values)
        X_lgb = pd.DataFrame(X_scaled, columns=FINAL_FEATURE_COLS)
        dtest = xgb.DMatrix(X_scaled) 

        # Predict probabilities: XGBoost uses its best iteration count, others use full model
        prob_xgb = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration))[0] 
        prob_lgb = lgb_model.predict_proba(X_lgb)[:, 1][0]
        prob_cbt = cbt_model.predict_proba(X_clean)[:, 1][0] 
        prob_logreg = logreg_model.predict_proba(X_lgb)[:, 1][0] 
        
        # Ensemble and Risk Adjustment
        total_w = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
        avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt + BEST_W_LOGREG * prob_logreg) / total_w

        confidence = 2 * abs(avg_prob - 0.5)
        position = np.clip(confidence * ML_CONF_FACTOR, 0, 2)

        return float(position)

    except Exception as e:
        logger.warning(f"ML Error (date_id: {date_id}): {e}")
        return 0.0


# ===========================================================================
# SERVER
# ===========================================================================
logger.info("Starting server")
import kaggle_evaluation.default_inference_server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

logger.info("✅ Complete. ")

2025-12-13 09:52:42,353 - Loading data and splitting for validation...
2025-12-13 09:52:42,854 - ✅ Oracle Dictionary ready. Binary Target created.
2025-12-13 09:52:42,856 - Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:4.18, LGB:0.30, CAT:1.93
2025-12-13 09:52:42,856 - Using OPTUNA BEST ENSEMBLE (Trial 3258), Risk Factor: 6.6209
2025-12-13 09:52:42,857 - Using Strategic Stopping: XGB:200 (Val Check), LGB/CAT:5000 (Full Train)
2025-12-13 09:52:42,858 - Training ensemble (Expanded Feature Set)...
2025-12-13 09:52:54,269 - Features: 1187 (Restored Optimized Set)
2025-12-13 09:54:02,460 - XGBoost trained (Stopped at 22 trees).
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 wa

In [2]:
# FINAL VERSION - STABILIZED ENSEMBLE WITH OPTUNA FACTOR (v5)
import os
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import pandas.api.types
from itertools import product
import warnings
import logging
import lightgbm as lgb
import platform
import sys
from contextlib import contextmanager 

# --- Setup and Logging ---
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', module='lightgbm')

os.environ['CATBOOST_DRIVER_COMPATIBLE'] = '1'
os.environ['CATBOOST_QUIET_MODE'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')

# ===========================================================================
# WARNING SUPPRESSION CONTEXT MANAGER
# ===========================================================================
@contextmanager
def suppress_stderr():
    """Temporarily redirect stderr to devnull to suppress native C warnings."""
    original_stderr = sys.stderr
    try:
        # Redirect stderr to /dev/null
        with open(os.devnull, 'w') as f:
            sys.stderr = f
            yield
    finally:
        # Restore original stderr
        sys.stderr = original_stderr


# ===========================================================================
# OFFICIAL KAGGLE METRIC
# ===========================================================================
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = 'row_id') -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ValueError('Predictions must be numeric')

    sol = solution.copy()
    sol['position'] = submission['prediction'].values

    if sol['position'].max() > MAX_INVESTMENT:
        raise ValueError(f'Position exceeds {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ValueError(f'Position below {MIN_INVESTMENT}')

    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    strategy_excess = sol['strategy_returns'] - sol['risk_free_rate']
    strategy_cum = (1 + strategy_excess).prod()
    strategy_mean = strategy_cum ** (1 / len(sol)) - 1
    strategy_std = sol['strategy_returns'].std()
    trading_days = 252

    if strategy_std == 0:
        return 0.0
    sharpe = strategy_mean / strategy_std * np.sqrt(trading_days)

    strategy_vol = float(strategy_std * np.sqrt(trading_days) * 100)

    market_excess = sol['forward_returns'] - sol['risk_free_rate']
    market_cum = (1 + market_excess).prod()
    market_mean = market_cum ** (1 / len(sol)) - 1
    market_std = sol['forward_returns'].std()
    market_vol = float(market_std * np.sqrt(trading_days) * 100)

    if market_vol == 0:
        return 0.0

    excess_vol = max(0, strategy_vol / market_vol - 1.2)
    vol_penalty = 1 + excess_vol

    return_gap = max(0, (market_mean - strategy_mean) * 100 * trading_days)
    return_penalty = 1 + (return_gap ** 2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


# ===========================================================================
# FEATURE ENGINEERING FUNCTION (POLARS - RESTORED ORIGINAL SET)
# ===========================================================================
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df_copy = df.clone()
    potential_base_feature_prefixes = ('M','E','I','P','V','S')
    all_potential_features = [
        c for c in df_copy.columns
        if c.startswith(potential_base_feature_prefixes) and c != 'market_forward_excess_returns'
    ]
    casting_expressions = []
    for c in all_potential_features:
        casting_expressions.append(
            pl.col(c).cast(pl.Float64, strict=False).alias(c)
        )
    if casting_expressions:
        df_copy = df_copy.with_columns(casting_expressions)
    base_features = [
        c for c in all_potential_features
        if df_copy.schema.get(c) in pl.NUMERIC_DTYPES
    ]
    expressions = []

    # --- 1. Lags (Original: 4 Lags) ---
    for c in base_features:
        for lag in [1, 2, 5, 10]:
            expressions.append(
                pl.col(c).shift(lag).over('date_id').fill_null(0).alias(f'{c}_L{lag}')
            )
    
    # --- 2. Rolling Window Features (Original: 2 Windows, 3 Stats) ---
    for c in base_features:
        for w in [5, 10]:
            expressions.append(
                pl.col(c).rolling_mean(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMean{w}')
            )
            expressions.append(
                pl.col(c).rolling_std(window_size=w, min_periods=1).over('date_id').fill_nan(0).fill_null(0).alias(f'{c}_RStd{w}')
            )
            expressions.append(
                pl.col(c).rolling_max(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMax{w}')
            )
            
    df_copy = df_copy.with_columns(expressions)
    expressions = []
    
    # --- 3. Rank and Z-Score ---
    for c in base_features:
        expressions.append(
            pl.col(c).rank(method='min').over('date_id').fill_null(0).alias(f'{c}_RANK')
        )
        mean_c = pl.col(c).mean().over('date_id')
        std_c  = pl.col(c).std().over('date_id')
        std_c_safe_expr = pl.when(pl.col(c).is_null() | std_c.is_null() | (std_c == 0)).then(1e-6).otherwise(std_c)
        expressions.append(
            ((pl.col(c).fill_null(mean_c) - mean_c) / std_c_safe_expr).fill_nan(0).fill_null(0).alias(f'{c}_ZSCORE')
        )
    df_copy = df_copy.with_columns(expressions)
    rank_cols = [f'{c}_RANK' for c in base_features]
    zscore_cols = [f'{c}_ZSCORE' for c in base_features]
    expressions = []

    # --- 4. Interactions (Original Set only) ---
    
    # Original Targeted Interactions (M4, M1, E1, V2)
    for c in ['M4','M1','E1','V2']:
        r = f'{c}_RANK'
        if f'{c}' in df_copy.columns and r in df_copy.columns:
             expressions.append((pl.col(c) * pl.col(r)).alias(f'{c}_x_{r}'))
             expressions.append((pl.col(c) / (pl.col(r) + 1e-6)).alias(f'{c}_div_{r}'))

    target_rank_cols = rank_cols[:12]
    # 1. Rank * Rank (Unique Pairs) - Adds 66 features
    for r_col1, r_col2 in product(target_rank_cols, target_rank_cols):
        c1 = r_col1.split('_')[0]
        c2 = r_col2.split('_')[0]
        if c1 < c2:
            expressions.append((pl.col(r_col1) * pl.col(r_col2)).fill_nan(0).fill_null(0).alias(f'{c1}R_x_{c2}R'))
    # 2. Rank * ZScore - Adds 9 features
    target_zscore_cols = zscore_cols[:9]
    for r_col, z_col in zip(rank_cols[:9], target_zscore_cols):
        expressions.append((pl.col(r_col) * pl.col(z_col)).fill_nan(0).fill_null(0).alias(f'{r_col}_x_{z_col}'))

    if expressions:
        df_copy = df_copy.with_columns(expressions)
    return df_copy


# ===========================================================================
# DATA LOADING AND SPLITTING
# ===========================================================================
logger.info("Loading data and splitting for validation...")
try:
    train_full_pl = pl.read_csv(DATA_PATH / "train.csv")
except FileNotFoundError:
    logger.error('Could not find \'train.csv\'. Please ensure the DATA_PATH is correct.')
    raise

train_full_pd = train_full_pl.to_pandas()

split_idx = int(len(train_full_pd) * 0.8)
train_pd = train_full_pd.head(split_idx).copy()
val_pd   = train_full_pd.tail(len(train_full_pd) - split_idx).copy()
full_train_pd = train_full_pd.copy() 

# --- Revert to Classification Target (Required for Optimal Score) ---
train_pd['target_binary'] = (train_pd['market_forward_excess_returns'] > 0).astype(np.int8)
val_pd['target_binary'] = (val_pd['market_forward_excess_returns'] > 0).astype(np.int8)
full_train_pd['target_binary'] = (full_train_pd['market_forward_excess_returns'] > 0).astype(np.int8)


train_pl = pl.from_pandas(train_pd)
val_pl   = pl.from_pandas(val_pd)
full_train_pl = pl.from_pandas(full_train_pd)

logger.info(f"✅ Oracle Dictionary ready. Binary Target created.")


# ===========================================================================
# ENSEMBLE PARAMETERS (UPDATED TO OPTUNA TRIAL 3258 - Score 1.06937)
# ===========================================================================

# --- OPTUNA TRIAL 3258 BEST (Score 1.06937) ---
BEST_W_XGB = 4.183022496349415     # XGBoost Weight
BEST_W_LGB = 0.29640061148676144   # LightGBM Weight
BEST_W_CAT = 1.9283230458964578   # CatBoost Weight
BEST_W_LOGREG = 0.0 
# --- **FIX: Restored Original Optuna Risk Confidence Factor** ---
ML_CONF_FACTOR = 6.6208527134892385 

# Model-specific Hyperparameters (Trial 3258)
XGB_MAX_DEPTH = 14
XGB_LR = 0.03260054929298955
XGB_REG_ALPHA = 0.1722628424051081
XGB_REG_LAMBDA = 10.528253815338909
XGB_SUBSAMPLE = 0.8018511121220447
XGB_COLSAMPLE = 0.7282877621162328

LGB_MAX_DEPTH = 8
LGB_LR = 0.013960942809215957
LGB_NUM_LEAVES = 141
LGB_L1 = 0.9245109459111609

CBT_DEPTH = 11
CBT_LR = 0.03679836629440696
CBT_L2_REG = 4.601313084093791

# Fixed Estimators and Stopping
N_ESTIMATORS_MAX = 5000
# Strategic Stopping parameters
XGB_ES = 200 # Use a moderate patience for the dominant model
LGB_CAT_ES = N_ESTIMATORS_MAX # Ensure full train for unstable models


logger.info(f"Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:{BEST_W_XGB:.2f}, LGB:{BEST_W_LGB:.2f}, CAT:{BEST_W_CAT:.2f}")
logger.info(f"Using RESTORED Optuna Risk Factor: {ML_CONF_FACTOR:.4f}")
logger.info(f"Using Strategic Stopping: XGB:{XGB_ES} (Val Check), LGB/CAT:{LGB_CAT_ES} (Full Train)")


# ===========================================================================
# ENSEMBLE PIPELINE (MAIN PIPELINE)
# ===========================================================================
logger.info("Training ensemble (Expanded Feature Set)...")

# 1. Feature Engineering 
train_engineered_pl = create_features(train_pl)
val_engineered_pl = create_features(val_pl)
full_train_engineered_pl = create_features(full_train_pl)

# 2. Feature Column Selection 
feature_cols = []
original_suffixes = ('_L1', '_L2', '_L5', '_L10', '_RMean5', '_RStd5', '_RMax5', '_RMean10', '_RStd10', '_RMax10', '_RANK', '_ZSCORE', '_x_RANK', '_div_RANK', '_x_R', '_x_ZSCORE')

for col in train_engineered_pl.columns:
    is_base_feature = col.startswith(('M', 'E', 'I', 'P', 'V', 'S'))
    is_engineered_feature = any(col.endswith(s) for s in original_suffixes)
    
    if (is_base_feature or is_engineered_feature) and (train_engineered_pl[col].null_count() / len(train_engineered_pl) < 0.95):
        feature_cols.append(col)

feature_cols = [c for c in feature_cols if c not in ['market_forward_excess_returns', 'target_binary', 'forward_returns', 'date_id']]
FINAL_FEATURE_COLS = feature_cols 
logger.info(f"Features: {len(FINAL_FEATURE_COLS)} (Restored Optimized Set)") 

# 3. Data Preparation for ML
# XGBoost Data (80% train, 20% val for ES)
X_train = train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_train = train_pl['target_binary'].fill_null(0).to_numpy().astype(int)
X_val = val_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_val = val_pl['target_binary'].fill_null(0).to_numpy().astype(int)

# LGB/CAT Data (100% full train set)
X_full_train = full_train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_full_train = full_train_pl['target_binary'].fill_null(0).to_numpy().astype(int)


# Enforce Feature Order
X_train = X_train.loc[:, FINAL_FEATURE_COLS]
X_val = X_val.loc[:, FINAL_FEATURE_COLS]
X_full_train = X_full_train.loc[:, FINAL_FEATURE_COLS]


# Scaling (Fit only on 80% train data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_full_train_scaled = scaler.transform(X_full_train) 

# DataFrames for LGB and DMatrix for XGB
X_lgb = pd.DataFrame(X_train_scaled, columns=FINAL_FEATURE_COLS)
X_val_lgb = pd.DataFrame(X_val_scaled, columns=FINAL_FEATURE_COLS)
X_full_lgb = pd.DataFrame(X_full_train_scaled, columns=FINAL_FEATURE_COLS)

dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dval = xgb.DMatrix(X_val_scaled, label=y_val)


# 4. Model Training (3 Models + 1 inert model)
# --- WRAP MODEL TRAINING IN suppress_stderr() TO HIDE C-LEVEL WARNINGS ---
with suppress_stderr():
    # --- XGBoost (Trained on 80% data with ES) ---
    xgb_params = {
        'max_depth': XGB_MAX_DEPTH, 'eta': XGB_LR, 'alpha': XGB_REG_ALPHA, 'lambda': XGB_REG_LAMBDA, 
        'subsample': XGB_SUBSAMPLE, 'colsample_bytree': XGB_COLSAMPLE, 'seed': 42,
        'objective': 'binary:logistic', 'tree_method': 'hist', 'eval_metric': 'logloss', 'verbosity': 0
    }

    xgb_model = xgb.train(
        xgb_params, dtrain, N_ESTIMATORS_MAX, evals=[(dval, 'val')],
        early_stopping_rounds=XGB_ES, verbose_eval=False
    )
    logger.info(f"XGBoost trained (Stopped at {xgb_model.best_iteration} trees).")

    # --- LightGBM (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    lgb_model = LGBMClassifier(
        n_estimators=N_ESTIMATORS_MAX, max_depth=LGB_MAX_DEPTH, learning_rate=LGB_LR, 
        num_leaves=int(LGB_NUM_LEAVES), lambda_l1=LGB_L1, random_state=123,
        objective='binary', metric='binary_logloss', n_jobs=-1, device_type='gpu',
        verbose=-1, _log_period=-1
    ).fit(
        X_full_lgb, y_full_train, 
    )
    logger.info(f"LGBMClassifier trained (Full 100% data, {lgb_model.n_estimators} trees).")


    # --- CatBoost (Trained on 100% data, NO ES/VALIDATION CHECK) ---
    cbt_model = CatBoostClassifier(
        iterations=N_ESTIMATORS_MAX, depth=int(CBT_DEPTH), learning_rate=CBT_LR, 
        l2_leaf_reg=CBT_L2_REG, random_seed=789, loss_function='Logloss', eval_metric='Logloss',
        bootstrap_type='Bayesian', task_type="GPU", logging_level='Silent', allow_writing_files=False, 
    ).fit(
        X_full_train, y_full_train, 
    )
    logger.info(f"CatBoostClassifier trained (Full 100% data, {cbt_model.get_best_iteration()} iterations).")

    # --- Logistic Regression (Trained on 80% data, but weighted 0.0) ---
    logreg_model = LogisticRegression(
        penalty='l2', C=0.01, solver='liblinear', max_iter=1000, random_state=999
    ).fit(X_lgb, y_train)
    logger.info(f"LogisticRegression trained.")

logger.info("✅ Ensemble trained.")
# --------------------------------------------------------------------------


# ===========================================================================
# VALIDATION – SCORE INTEGRATION (3-MODEL ENSEMBLE)
# ===========================================================================
logger.info('Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...')

# Prediction for XGBoost uses its early-stopped version (best_iteration) on the 20% validation set
prob_xgb = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration)) 
# Prediction for LGB/Cat uses the full 5000-iteration model on the 20% validation set
prob_lgb = lgb_model.predict_proba(X_val_lgb)[:, 1]
prob_cbt = cbt_model.predict_proba(X_val)[:, 1]
prob_logreg = logreg_model.predict_proba(X_val_lgb)[:, 1]

# Calculate final ensemble position using the 3 positive weights
total_w_final = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt + BEST_W_LOGREG * prob_logreg) / total_w_final

confidence = 2 * np.abs(avg_prob - 0.5)
positions_final = np.clip(confidence * ML_CONF_FACTOR, 0.0, 2.0)

submission_df_final = pd.DataFrame({'prediction': positions_final})

try:
    real_ps = score(val_pd, submission_df_final)
    logger.info(f'FINAL SUBMISSION PS SCORE ON VALIDATION (STABILIZED, OPTUNA FACTOR) = {real_ps:.6f}') 
except Exception as e:
    logger.error(f'Final Scoring error: {e}')
    real_ps = 0.0


# ===========================================================================
# PREDICT FUNCTION (FINAL ROBUST VERSION - 3-MODEL ENSEMBLE)
# ===========================================================================
def predict(test: pl.DataFrame) -> float:
    
    # Define constants locally for the predict function scope
    BEST_W_XGB = 4.183022496349415
    BEST_W_LGB = 0.29640061148676144
    BEST_W_CAT = 1.9283230458964578
    BEST_W_LOGREG = 0.0 
    ML_CONF_FACTOR = 6.6208527134892385 # Restored Optuna factor for prediction
    
    if xgb_model is None or lgb_model is None or cbt_model is None or logreg_model is None or scaler is None or not FINAL_FEATURE_COLS:
        return 0.0

    date_id = None
    try:
        date_id = int(test.select("date_id").to_series().item())
    except:
        pass
    
    try:
        test_engineered = create_features(test)
        X_test = test_engineered.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
        
        X_clean = X_test.loc[:, FINAL_FEATURE_COLS]
        X_clean = X_clean.fillna(0).replace([np.inf, -np.inf], 0) 

        X_scaled = scaler.transform(X_clean.values)
        X_lgb = pd.DataFrame(X_scaled, columns=FINAL_FEATURE_COLS)
        dtest = xgb.DMatrix(X_scaled) 

        # Predict probabilities: XGBoost uses its best iteration count, others use full model
        prob_xgb = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration))[0] 
        prob_lgb = lgb_model.predict_proba(X_lgb)[:, 1][0]
        prob_cbt = cbt_model.predict_proba(X_clean)[:, 1][0] 
        prob_logreg = logreg_model.predict_proba(X_lgb)[:, 1][0] 
        
        # Ensemble and Risk Adjustment
        total_w = BEST_W_XGB + BEST_W_LGB + BEST_W_CAT + BEST_W_LOGREG
        avg_prob = (BEST_W_XGB * prob_xgb + BEST_W_LGB * prob_lgb + BEST_W_CAT * prob_cbt + BEST_W_LOGREG * prob_logreg) / total_w

        confidence = 2 * abs(avg_prob - 0.5)
        position = np.clip(confidence * ML_CONF_FACTOR, 0, 2)

        return float(position)

    except Exception as e:
        logger.warning(f"ML Error (date_id: {date_id}): {e}")
        return 0.0


# ===========================================================================
# SERVER
# ===========================================================================
logger.info("Starting server")
import kaggle_evaluation.default_inference_server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

logger.info("✅ Complete. ")

2025-12-13 10:50:55,482 - Loading data and splitting for validation...
2025-12-13 10:50:55,775 - ✅ Oracle Dictionary ready. Binary Target created.
2025-12-13 10:50:55,776 - Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:4.18, LGB:0.30, CAT:1.93
2025-12-13 10:50:55,777 - Using RESTORED Optuna Risk Factor: 6.6209
2025-12-13 10:50:55,778 - Using Strategic Stopping: XGB:200 (Val Check), LGB/CAT:5000 (Full Train)
2025-12-13 10:50:55,778 - Training ensemble (Expanded Feature Set)...
2025-12-13 10:51:10,168 - Features: 1187 (Restored Optimized Set)
2025-12-13 10:52:17,658 - XGBoost trained (Stopped at 22 trees).
2025-12-13 10:57:05,942 - LGBMClassifier trained (Full 100% data, 5000 trees).
2025-12-13 11:49:59,883 - CatBoostClassifier trained (Full 100% data, None iterations).
2025-12-13 11:50:01,577 - LogisticRegression trained.
2025-12-13 11:50:01,578 - ✅ Ensemble trained.
2025-12-13 11:50:01,579 - Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...
2025-12-13 1

2025-12-05 11:13:17,668 - Loading data and splitting for validation...
2025-12-05 11:13:17,882 - ✅ Oracle Dictionary ready. Binary Target created.
2025-12-05 11:13:17,883 - Using OPTUNA BEST ENSEMBLE (Trial 3258), Weights: XGB:4.18, LGB:0.30, CAT:1.93
2025-12-05 11:13:17,884 - Using OPTUNA BEST ENSEMBLE (Trial 3258), Risk Factor: 6.6209
2025-12-05 11:13:17,884 - Training ensemble (Expanded Feature Set)...
2025-12-05 11:13:24,775 - Features: 1187 (Restored Optimized Set)
2025-12-05 11:13:46,677 - XGBoost trained (Stopped at 18 trees).
2025-12-05 11:13:50,384 - LGBMClassifier trained (Stopped at 1 trees).
2025-12-05 11:14:29,072 - CatBoostClassifier trained (Stopped at 3 iterations).
2025-12-05 11:14:30,877 - LogisticRegression trained.
2025-12-05 11:14:30,877 - ✅ Ensemble trained.
2025-12-05 11:14:30,878 - Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...
2025-12-05 11:14:30,923 - FINAL SUBMISSION PS SCORE ON VALIDATION (3-MODEL ENSEMBLE) = 1.069377
2025-12-05 11:14:30,924 - Starting server
2025-12-05 11:14:32,920 - ✅ Complete. 

In [3]:
import pandas as pd
import numpy as np
import os

submission_path = '/kaggle/working/submission.parquet'

## 📊 SUBMISSION FILE VALIDATION

# Check if the file was successfully created
if not os.path.exists(submission_path):
    print(f"Validation Error: Submission file not found at {submission_path}")
    print("NOTE: This file is created during the 'Running local gateway for testing...' step.")
else:
    try:
        # Read the submission file
        df_sub = pd.read_parquet(submission_path)

        # --- Validation Checks ---

        # 1. Identify the prediction column (the single float column)
        float_cols = df_sub.select_dtypes(include=[np.float64]).columns

        if len(float_cols) == 1:
            prediction_col_name = float_cols[0]
            print(f"Column Check: Found single prediction column named '{prediction_col_name}'.")
        else:
            prediction_col_name = 'allocation' # Use standard name for range check fallback
            print(f"Column Check: Found {len(float_cols)} float columns. Using '{prediction_col_name}' for range check.")

        # 2. Check allocation range (0.0 to 2.0)
        if prediction_col_name in df_sub.columns:
            min_val = df_sub[prediction_col_name].min()
            max_val = df_sub[prediction_col_name].max()

            # Check if all values are between 0.0 and 2.0 (inclusive)
            if min_val >= 0.0 and max_val <= 2.0:
                range_check = "PASS"
            else:
                range_check = f"FAIL (Min: {min_val:.4f}, Max: {max_val:.4f})"

            print(f"Allocation Range Check (0.0 to 2.0): {range_check}")

        # 3. Display the file info
        print("\nFirst 5 Rows of Submission:")
        print(df_sub.head())

        print("\nSubmission Info:")
        df_sub.info()

    except Exception as e:
        print(f"Validation Error: Could not read or process the Parquet file. Error: {e}")

Column Check: Found single prediction column named 'prediction'.
Allocation Range Check (0.0 to 2.0): PASS

First 5 Rows of Submission:
   date_id  prediction
0     8980    0.569003
1     8981    0.898077
2     8982    1.284482
3     8983    1.016433
4     8984    0.162562

Submission Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date_id     10 non-null     int64  
 1   prediction  10 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 292.0 bytes
